In [ ]:

#Backward Chaining 

import csv
import random
from collections import Counter
import time





columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"] #all classes in car.data file order
classes = ["unacc", "acc", "good", "vgood"] # 4 possible classes

with open("car.data") as carFile: #opens car.data as carFile
    carRows = [dict(zip(columns, row)) for row in csv.reader(carFile) if row] # zip labels each value with its column name and then is stored as a dict
#if row skips any blank rows
random.seed(21) #fixed seed so test/train split is reproducible
shuffled = carRows[:] #copy so the original is kept
random.shuffle(shuffled) #randomise the dataset
cut = int(0.7 * len(shuffled)) #splits the shuffled data set in 70%-30%
train, test = shuffled[:cut], shuffled[cut:] #train = 70%, 30% = test

majority = Counter(r["class"] for r in train).most_common(1)[0][0] #counts each class in train. lists most common and first in list and first in tuple class
baseline = sum(r["class"] == majority for r in test) / len(test) #uses most common class as default
print(f"Baseline: {baseline:.2f}") #prints output of baseline to the second decimal


rules = [ 
    ({"safety=low"}, "unacc"),
    ({"persons=2"}, "unacc"),
    ({"doors=2"}, "unacc"),
    ({"lug_boot=small"}, "unacc"),

    
    ({"safety=med", "lug_boot=med"}, "acc"),
    ({"safety=med", "lug_boot=big"}, "acc"),
    ({"safety=med", "doors=4"}, "acc"),
    ({"safety=med", "doors=5more"}, "acc"),
    ({"safety=med", "persons=4"}, "acc"),
    ({"safety=med", "persons=more"}, "acc"),

    ({"safety=high", "doors=5more", "persons=more", "lug_boot=big"}, "vgood"),
    
    ({"safety=high", "doors=4"}, "good"),
    ({"safety=high", "doors=5more"}, "good"),
    ({"safety=high", "persons=4"}, "good"),
    ({"safety=high", "persons=more"}, "good"),
    ({"safety=high", "lug_boot=med"}, "good"),
    ({"safety=high", "lug_boot=big"}, "good"),
    
    
    
]




def back_chain(goal, facts, rules):
    if goal in facts: #the goal is already a car's facts so already proved
        return True
    for conditions, conclusion in rules: #if goal is not in the car's facts, look for another rule to prove it
        if conclusion == goal: #check only rules where the conclusion is the goal
            if all(back_chain(c, facts, rules) for c in sorted(conditions)): #only continue when each condidtion is proved as its own subgoal, false if even one conditon is false
                return True
    return False #no rule is applicable so goal cannot be proved

def prove(goal, facts, rules, depth=0): #idnentical logic as prior but, shows each step of the process for performance metric
    indent = "  " * depth #showcases the and indents subgoals prior the goals that triggered said subgoals
    if goal in facts: #
        print(f"{indent}{goal}: known fact") #prints to user the goal is a known fact
        return True
    for conditions, conclusion in rules: #looks for rules that conclude the goal
        if conclusion == goal: 
            print (f"{indent}{goal}: try rule needing {sorted(conditions)}") #shows user which rule is being tried and informs which rules are also needed
            if all(prove(c, facts, rules, depth + 1) for c in sorted(conditions)): #tries to prove each condition as a sub goal and needs every one to succeed
                print(f"{indent}{goal}: proved") #if every condtion was proved
                return True
    print(f"{indent}{goal}: cannot be proved") #else, rule was not appropriate and couldnt be proved
    return False


def predict(row):
    facts = {f"{a}={v}" for a, v in row.items() if a in ("doors", "persons", "lug_boot", "safety")}  # row.items pairs attributes to values and drops "class", as thats the end result of the prediction
    for goal in ["unacc", "acc", "vgood", "good"]: #try each class as a goal. First mtach wins
        if back_chain(goal, facts, rules): #if the goal can be proved from the factss:
            return goal #returns the first match as the final prediction
    return majority #falls back to "unacc"

train_accuracy = sum(predict(r) == r["class"] for r in train) / len(train)  
print(f"Train accuracy: {train_accuracy:.2f}")

begin = time.perf_counter() #declares beginning of runtime

accuracy = sum(predict(r) == r["class"] for r in test) / len(test) #each prediction counts as 1, sum() combines all tallies and divides by total test car rows
end = time.perf_counter() #declares end of runtime
print(f"Accuracy: {accuracy:.2f}") #prints test split accuracy to the second decimal
print(f"Baseline: {baseline:.2f}") #reprints baseline for clarity in terminal
print(f"Runtime: {end - begin:.6f}") #calculates total runtime of accuracy calculation. Inspired from: https://www.geeksforgeeks.org/python/time-perf_counter-function-in-python/

row = test[57] #test row of an individual car
facts = {f"{a}={v}" for a, v in row.items() if a in ("doors", "persons", "lug_boot", "safety")}
print("test car: ") #seperates test car in terminal
prove("unacc", facts, rules) #prints each step in predicting desired class






Baseline: 0.71
Train accuracy: 0.73
Accuracy: 0.72
Baseline: 0.71
Runtime: 0.002910
test car: 
unacc: try rule needing ['safety=low']
  safety=low: known fact
unacc: proved


True

In [ ]:
#Forward Chaining

import csv
import random
from collections import Counter
import time




columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"] #all classes in car.data file order
classes = ["unacc", "acc", "good", "vgood"] # 4 possible classes

with open("car.data") as carFile: #opens car.data as carFile
    carRows = [dict(zip(columns, row)) for row in csv.reader(carFile) if row] # zip labels each value with its column name and then is stored as a dict
    #if row skips any blank rows 

random.seed(21) #fixed seed so test/train split is reproducible
shuffled = carRows[:] #copy so the original is kept
random.shuffle(shuffled) #randomise the dataset
cut = int(0.7 * len(shuffled)) #splits the shuffled data set in 70%-30%
train, test = shuffled[:cut], shuffled[cut:] #train = 70%, 30% = test

majority = Counter(r["class"] for r in train).most_common(1)[0][0] #counts each class in train. lists most common and first in list and first in tuple class
baseline = sum(r["class"] == majority for r in test) / len(test) #accuracy if "unacc" was guessed
print(f"Baseline: {baseline:.2f}") #prints output of baseline to the second decimal


rules = [ 
    ({"safety=low"}, "unacc"),
    ({"persons=2"}, "unacc"),
    ({"doors=2"}, "unacc"),
    ({"lug_boot=small"}, "unacc"),

    
    ({"safety=med", "lug_boot=med"}, "acc"),
    ({"safety=med", "lug_boot=big"}, "acc"),
    ({"safety=med", "doors=4"}, "acc"),
    ({"safety=med", "doors=5more"}, "acc"),
    ({"safety=med", "persons=4"}, "acc"),
    ({"safety=med", "persons=more"}, "acc"),

    ({"safety=high", "doors=5more", "persons=more", "lug_boot=big"}, "vgood"),
    
    ({"safety=high", "doors=4"}, "good"),
    ({"safety=high", "doors=5more"}, "good"),
    ({"safety=high", "persons=4"}, "good"),
    ({"safety=high", "persons=more"}, "good"),
    ({"safety=high", "lug_boot=med"}, "good"),
    ({"safety=high", "lug_boot=big"}, "good"),
    
    
    
]



def for_chain_trace(facts, rules):
    known, trace = set(facts), [] #known is the starting facts and trace is the rules fired
    changed = True #ensure the loop runs atleast once
    while changed:
        changed = False
        for conditions, conclusion in rules: #list order of every rule
            if conditions <= known and conclusion not in known: #fire if every condition is true and conclusion is new
                known.add(conclusion) #fires rule, conclusion is now a fact
                trace.append((sorted(conditions), conclusion))# stores fired rule. "sorted" sorts  alphabetically
                changed = True #new rule fired so loop
    return known, trace #once new rules stop firing, return

row = test[57] #test row of an individual car
facts = {f"{a}={v}" for a, v in row.items() if a in ("doors", "persons", "lug_boot", "safety")}

known, trace = for_chain_trace(facts, rules) #runs forward chaining on car's facts
for conditions, conclusion in trace: #
    print(f"{' + '.join(conditions)} -> {conclusion}") #the cars actual class compared with what fired


def predict(row):
    facts = {f"{a}={v}" for a, v in row.items() if a in ("doors", "persons", "lug_boot", "safety")} # row.items pairs attributes to values and drops "class" as thats the end result
    known, trace = for_chain_trace(facts, rules) #runs forward training on the car's facts
    for conditions, conclusion in trace: #trace is the log of rules fired, follows from first to last
        if conclusion in classes: #skips any conclusion that isnt a class, not utilised
            return conclusion #returns first class as conclusion
    return majority # if no conclusion reached, use majority as default "unacc" in this case
train_accuracy = sum(predict(r) == r["class"] for r in train) / len(train)   
print(f"Train accuracy: {train_accuracy:.2f}")
begin = time.perf_counter() #declares beginning of runtime
accuracy = sum(predict(r) == r["class"] for r in test) / len(test) #each prediction counts as 1, sum() combines all tallies
#prediction id then divided by length of test data to create accuracy
end = time.perf_counter() #declares end of runtime
print(f"Accuracy: {accuracy:.2f}") #prints accuracy to the second decimal
print(f"Runtime: {end - begin:.6f}")  #calculates total runtime of accuracy calculation. Inspired from: https://www.geeksforgeeks.org/python/time-perf_counter-function-in-python/





Baseline: 0.71
safety=low -> unacc
Train accuracy: 0.73
Accuracy: 0.72
Runtime: 0.002142
